In [ ]:
# Cell 1: Instalação de dependências
!pip install -q transformers datasets peft accelerate bitsandbytes trl torch
!pip install -q rouge-score nltk pandas numpy matplotlib scikit-learn

In [ ]:
# Cell 2: Imports e configurações
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset, load_dataset
from peft import LoraConfig, get_peft_model
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import random
import os
os.environ["WANDB_DISABLED"] = "true"

# Configurações
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
DATASET_NAME = "tatsu-lab/alpaca"
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
# Cell 3: Carregar modelo e tokenizador
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"Modelo {MODEL_NAME} carregado")
print(f"Arquitetura do modelo: {model.config.model_type}")

In [ ]:
# Cell 4: Estratégia de divisão de dados OTIMIZADA (tamanho reduzido)
def format_instruction(sample):
    return {
        "text": f"### Instrução:\n{sample['instruction']}\n\n### Entrada:\n{sample['input']}\n\n### Resposta:\n{sample['output']}"
    }

# Carregar dataset
dataset = load_dataset(DATASET_NAME)
dataset = dataset.map(format_instruction)

# Converter para DataFrame para facilitar manipulação
df = pd.DataFrame(dataset['train'])

print(f"Dataset original: {len(df)} exemplos")

# Amostrar apenas 20% dos dados para treinamento + validação
reduced_df = df.sample(frac=0.2, random_state=SEED)  # ⬇️ Reduz para 20% do dataset
print(f"Dataset após redução: {len(reduced_df)} exemplos")

# Divisão em treino/validação/teste com tamanhos balanceados
train_df, temp_df = train_test_split(
    reduced_df,
    test_size=0.3,  # 30% para validação + teste
    random_state=SEED,
    stratify=reduced_df['instruction'].apply(lambda x: hash(x) % 5)
)

eval_df, test_df = train_test_split(
    temp_df,
    test_size=0.33,  # 33% do temp (10% total) para teste
    random_state=SEED
)

# Converter de volta para Dataset
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
eval_dataset = Dataset.from_pandas(eval_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

print(f"Dataset final:")
print(f"   Treino: {len(train_dataset)} exemplos")
print(f"   Validação: {len(eval_dataset)} exemplos")
print(f"   Teste: {len(test_dataset)} exemplos")
print(f"   Total: {len(train_dataset) + len(eval_dataset) + len(test_dataset)} exemplos")

# Mostrar distribuição
print(f"\n Distribuição:")
print(f"   Treino: {len(train_dataset)/len(reduced_df)*100:.1f}%")
print(f"   Validação: {len(eval_dataset)/len(reduced_df)*100:.1f}%")
print(f"   Teste: {len(test_dataset)/len(reduced_df)*100:.1f}%")

In [ ]:
# Cell 5: Função de tokenização otimizada
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )

# Aplicar tokenização
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

In [ ]:
# Cell 6: Configuração LoRA

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Cell 7: Configuração do treinamento
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir="./qwen2.5-0.5b-sft-dolly",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    fp16=True,
    logging_steps=50,
    eval_steps=150,
    save_steps=300,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=8e-6,
    warmup_ratio=0.1,
    weight_decay=0.02,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print(" Configuração de treinamento otimizada!")
print(f" Épocas: {training_args.num_train_epochs} (aumentado para melhor aprendizado)")
print(f" Learning rate: {training_args.learning_rate} (ligeiramente ajustado)")
print(f" Avaliação a cada: {training_args.eval_steps} steps")
print(f" Salvamento a cada: {training_args.save_steps} steps")
print(f" Dataset de treino: {len(tokenized_train)} exemplos")
print(f" Dataset de validação: {len(tokenized_eval)} exemplos")

In [ ]:
# Cell 8: Treinamento
print("🚀 Iniciando treinamento otimizado...")

# Avaliação inicial
print("\n === Avaliação INICIAL ===")
initial_eval = trainer.evaluate()
initial_loss = initial_eval['eval_loss']
initial_ppl = np.exp(initial_loss)
print(f" Loss inicial: {initial_loss:.4f}")
print(f" Perplexidade inicial: {initial_ppl:.4f}")

# Treinamento
print("\n Iniciando treinamento...")
train_result = trainer.train()

# Salvar modelo final
trainer.save_model()
tokenizer.save_pretrained("./qwen2.5-0.5b-sft-dolly")

print("\n Treinamento concluído!")
print(f" Loss final de treinamento: {train_result.training_loss:.4f}")

In [ ]:
# Cell 9: Avaliação comparativa
def generate_response(model, tokenizer, prompt, max_length=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_length,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

try:
    ft_model = AutoModelForCausalLM.from_pretrained(
        "./qwen2.5-0.5b-sft-dolly",
        torch_dtype=torch.float16,
        device_map="auto"
    )
    ft_tokenizer = AutoTokenizer.from_pretrained("./qwen2.5-0.5b-sft-dolly")
    print(" Modelo fine-tuned carregado com sucesso!")
except Exception as e:
    print(f" Erro ao carregar modelo fine-tuned: {e}")
    print(" Usando modelo base para comparação...")
    ft_model = model
    ft_tokenizer = tokenizer

# Testar ambos os modelos
test_prompts = [
    "Explique o que é aprendizado de máquina:",
    "Como fazer uma omelete?",
]

print("COMPARAÇÃO: MODELO BASE vs MODELO FINE-TUNED")
print("=" * 80)

for i, prompt in enumerate(test_prompts):
    print(f"\n PROMPT {i+1}: {prompt}")
    print("-" * 50)

    # Modelo base
    try:
        base_response = generate_response(model, tokenizer, prompt)
        # Extrai apenas a parte da resposta
        base_answer = base_response.replace(prompt, "").strip()
        print(f" MODELO BASE:\n{base_answer[:300]}..." if len(base_answer) > 300 else base_answer)
    except Exception as e:
        print(f" Erro no modelo base: {e}")

    print()

    # Modelo fine-tuned
    try:
        ft_response = generate_response(ft_model, ft_tokenizer, prompt)
        # Extrair apenas a parte da resposta
        ft_answer = ft_response.replace(prompt, "").strip()
        print(f" MODELO FINE-TUNED:\n{ft_answer[:300]}..." if len(ft_answer) > 300 else ft_answer)
    except Exception as e:
        print(f" Erro no modelo fine-tuned: {e}")

    print("-" * 50)

In [ ]:
# Cell 10: Avaliação quantitativa no conjunto de teste
from rouge_score import rouge_scorer

def calculate_rouge(predictions, references):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = []

    for pred, ref in zip(predictions, references):
        score = scorer.score(ref, pred)
        scores.append(score)

    return scores

# Gerar predições no conjunto de teste
print("Avaliando no conjunto de teste...")
test_predictions_base = []
test_predictions_ft = []
references = []

for example in test_dataset.select(range(50)):  # Amostra de 50 exemplos
    prompt = example['text'].split("### Resposta:")[0]  # Manter apenas o prompt
    reference = example['text'].split("### Resposta:")[1] if "### Resposta:" in example['text'] else ""

    pred_base = generate_response(model, tokenizer, prompt)
    pred_ft = generate_response(ft_model, ft_tokenizer, prompt)

    test_predictions_base.append(pred_base)
    test_predictions_ft.append(pred_ft)
    references.append(reference)

# Calcular métricas ROUGE
rouge_scores_base = calculate_rouge(test_predictions_base, references)
rouge_scores_ft = calculate_rouge(test_predictions_ft, references)

# Calcular médias
def average_rouge(scores):
    avg_scores = {}
    for key in ['rouge1', 'rouge2', 'rougeL']:
        avg_scores[key] = {
            'fmeasure': np.mean([s[key].fmeasure for s in scores]),
            'precision': np.mean([s[key].precision for s in scores]),
            'recall': np.mean([s[key].recall for s in scores])
        }
    return avg_scores

avg_base = average_rouge(rouge_scores_base)
avg_ft = average_rouge(rouge_scores_ft)

print("\n MÉTRICAS ROUGE (F1-Score):")
print(f"{'Metric':<10} {'Base':<10} {'Fine-tuned':<10} {'Diferença':<10}")
for key in ['rouge1', 'rouge2', 'rougeL']:
    base_f1 = avg_base[key]['fmeasure']
    ft_f1 = avg_ft[key]['fmeasure']
    diff = ft_f1 - base_f1
    print(f"{key:<10} {base_f1:.4f}    {ft_f1:.4f}    {diff:+.4f}")